# as-strided-windowing — worked example 1: Sliding-window sum over a 1-D tensor via as_strided

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `as-strided-windowing`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A length-`L` 1-D tensor can be re-viewed as a `(L-K+1, K)` window matrix with **zero copy** by reusing its element stride `s` on both axes: `stride=(s, s)`. The outer axis advances the window start by one element; the inner axis walks across the `K` elements of one window. Once you have the window view, reductions like a moving sum are just a `.sum(dim=-1)`.

## Worked solution

**Goal.** Compute, for every length-`K` contiguous window of `x`, the sum of its elements, using an `as_strided` view rather than a Python loop.

1. **Read the source stride.** `s, = x.stride()` gives the number of storage elements between consecutive logical elements of `x`. We must NOT assume `s == 1`: if `x` came from a slice like `source[::2]`, `s` would be `2`, and hard-coding `1` would read the wrong memory.
2. **Compute the output length.** With stride-1 windows there are `L_out = L - K + 1` valid window start positions (the last window starts at index `L-K`).
3. **Build the window view.** `t.as_strided(x, size=(L_out, K), stride=(s, s))`. Both strides are `s` because moving one window forward (`+1` on the outer axis) and moving one element within a window (`+1` on the inner axis) are the *same* one-element step in storage. This is a view — no data is copied.
4. **Reduce.** Summing over the last axis collapses each `K`-window into a scalar, giving the moving-window sum of shape `(L_out,)`.

**Why it works.** `as_strided` reinterprets the existing storage; the overlapping windows share memory, which is exactly what makes it zero-copy. Because we pulled `s` from the tensor itself, the view is correct even for non-contiguous inputs.

In [ ]:
def windowed_sum_1d(x: Tensor, K: int) -> Tensor:
    L = x.shape[0]
    s, = x.stride()
    L_out = L - K + 1
    windows = t.as_strided(x, size=(L_out, K), stride=(s, s))
    return windows.sum(dim=-1)

t.manual_seed(0)
x = t.arange(1, 8, dtype=t.float32)  # [1,2,3,4,5,6,7]
print(windowed_sum_1d(x, 3))         # [6, 9, 12, 15, 18]